In [1]:
import os
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
df = pd.read_csv("E:\SmartHealthPrediction\dataset\obesity.csv")

# Encode target variable
le = LabelEncoder()
df["NObeyesdad"] = le.fit_transform(df["NObeyesdad"])

# One-hot encode categorical features
df = pd.get_dummies(df, drop_first=True)

# Split features & target
X = df.drop("NObeyesdad", axis=1)
y = df["NObeyesdad"]

# Feature scaling
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Hyperparameter tuning
param_grid = {
    "n_estimators": [200, 300],
    "max_depth": [10, 15, None],
    "min_samples_split": [2, 4]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)

# Best model
model = grid.best_estimator_

# Evaluation
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("Best Parameters:", grid.best_params_)
print("Obesity Model Accuracy:", round(acc * 100, 2), "%")
print(classification_report(y_test, y_pred))

# Save model
pickle.dump(model, open("model/obesity_model.pkl", "wb"))
print("Obesity model saved successfully")

os.makedirs("model", exist_ok=True)
path = os.path.join("model", "obesity_model.pkl")

with open(path, "wb") as f:
    pickle.dump(model, f)

print("Obesity model saved")
print("File exists:", os.path.exists(path))
print("File size:", os.path.getsize(path), "bytes")


Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 300}
Obesity Model Accuracy: 94.33 %
              precision    recall  f1-score   support

           0       0.93      0.96      0.95       228
           1       0.96      0.92      0.94       195

    accuracy                           0.94       423
   macro avg       0.94      0.94      0.94       423
weighted avg       0.94      0.94      0.94       423

Obesity model saved successfully
Obesity model saved
File exists: True
File size: 9750121 bytes
